# Tarea Hogar 04 — extremos en la zona que si pago (`z496`)

Archivo aparte de `z495` (`HT4950`). Experimento **`HT4960`**. No pisa esos trials.

El puesto 1 de z495 (618M, familia `semilla`) era la config del centro con un split suertudo. En Kaggle dio **359.7** y perdio contra `KA5940` **375.9** en el cupo 10500. Rankear por el maximo de una semilla elige ruido.

Lo que en z495 se movio de verdad, y aca se lleva mas al extremo:

- hojas chicas (el punto Denicolay, 8) hacia **2–16**, no hacia 511
- `undersampling` hacia **1.0** (los buenos trials usaban el mes entero)
- `learning_rate` mas bajo con muchas mas rondas (hasta 5000)
- `max_bin` hacia **255–1023**
- `bagging_fraction` hacia **0.1–0.3** (familia estable, media alta)
- `min_data_in_leaf` fino entre 30 y 150
- `feature_fraction` hacia 0.08–0.3 y hacia 1

Cada config se evalua en las **mismas 3 semillas** y gana la **media**. `is_unbalance` no se repite (en z495 cayo a 122M).


## 0. Librerias


In [ ]:
# +++ lightgbm puede no estar en la imagen: se instala si falta
if (!require("data.table")) install.packages("data.table")
if (!require("lightgbm")) install.packages("lightgbm")
require("data.table")
require("lightgbm")
require("parallel")

setDTthreads(percent = 100)  # +++ el padre usa todos los cores para fread; los workers van a 1
options(scipen = 999)


## 1. PARAM


In [ ]:
PARAM <- list()
PARAM$semilla_primigenia <- 271211L   # +++ TU semilla
PARAM$semilla2 <- 552581L             # +++ semilla del undersampling (la "semilla2" de la planilla)
PARAM$estudiante <- "Maceo, Marcos"
PARAM$experimento <- "HT4960"
PARAM$mc_cores <- max(1L, detectCores() - 1L)  # +++ 7 workers en e2-highmem-8
PARAM$mc_cores


## 2. Dataset (solo 202107: ahi hay clase)


In [ ]:
# +++ deteccion de entorno: Colab / VM GCP / local
candidatos_exp <- c("/content/buckets/b1/exp", path.expand("~/buckets/b1/exp"), file.path(getwd(), "exp"))
base_exp <- candidatos_exp[dir.exists(candidatos_exp)][1]
if (is.na(base_exp)) {
  base_exp <- candidatos_exp[3]
  dir.create(base_exp, recursive = TRUE, showWarnings = FALSE)
}
dir.create(file.path(base_exp, PARAM$experimento), showWarnings = FALSE)
setwd(file.path(base_exp, PARAM$experimento))
getwd()

candidatos_ds <- c(
  "/content/datasets/dataset_pequeno.csv",
  path.expand("~/datasets/dataset_pequeno.csv"),
  path.expand("~/buckets/b1/datasets/dataset_pequeno.csv")
)
archivo_dataset <- candidatos_ds[file.exists(candidatos_ds)][1]
stopifnot(!is.na(archivo_dataset))

dataset <- fread(archivo_dataset)
# +++ solo el mes con clase. 202109 queda para el submit final, no para elegir hiperparametros
dataset_mes <- dataset[foto_mes == 202107]
dataset_mes[, c("clase01", "azar", "training") := NULL]
nrow(dataset_mes)
dataset_mes[, .N, clase_ternaria]


## 3. Un job

Entrena en el 70% (con undersampling de CONTINUA), predice el 30%, ganancia normalizada al mes completo. El corte fijo es `1/40`. `mejores_envios` es el cupo que maximiza la ganancia en ese holdout, escalado al mes entero (la columna de la planilla donde Denicolay puso 11000).


In [ ]:
# +++ un job = un modelo LightGBM. Ganancia en holdout 70/30 de 202107 (no AUC).
# +++ PSOCK: esta funcion tiene que ser autocontenida (no usa closures del padre).
eval_job <- function(job) {
  tryCatch({
    data.table::setDTthreads(1)
    t0 <- Sys.time()

    split_strat <- function(dt, p, seed) {
      set.seed(as.integer(seed))
      dwork <- data.table::copy(dt)
      dwork[, `:=`(azar = runif(.N), .rowid = seq_len(.N))]
      data.table::setorderv(dwork, c("clase_ternaria", "azar"))
      dwork[, fold := seq_len(.N) / .N, by = clase_ternaria]
      ids <- dwork[fold <= p, .rowid]
      list(
        train = dt[ids],
        test = dt[setdiff(seq_len(nrow(dt)), ids)]
      )
    }

    sp <- split_strat(dataset_mes, 0.70, job$semilla_split)
    dtrain <- sp$train
    dtest <- sp$test

    set.seed(as.integer(job$semilla2))
    dtrain[, azar_us := runif(.N)]
    dfit <- dtrain[clase_ternaria %in% c("BAJA+1", "BAJA+2") | azar_us <= job$undersampling]

    es <- as.integer(job$early_stopping_rounds)
    dval <- NULL
    if (es > 0L && nrow(dfit) > 1000L) {
      spv <- split_strat(dfit, 0.85, job$semilla_split + 17L)
      dval <- spv$test
      dfit <- spv$train
    }

    lab <- function(cl) {
      if (identical(job$label_mode, "baja1y2")) {
        as.integer(cl %in% c("BAJA+1", "BAJA+2"))
      } else {
        as.integer(cl == "BAJA+2")
      }
    }

    drop_cols <- c(
      "clase_ternaria", "numero_de_cliente", "foto_mes",
      "azar", "azar_us", "fold", ".rowid"
    )
    campos <- setdiff(colnames(dfit), drop_cols)
    num_ok <- vapply(dfit[, campos, with = FALSE], is.numeric, logical(1))
    campos <- campos[num_ok]

    mat_tr <- data.matrix(dfit[, campos, with = FALSE])
    y <- lab(dfit$clase_ternaria)
    ds <- lightgbm::lgb.Dataset(
      data = mat_tr,
      label = y,
      params = list(
        max_bin = as.integer(job$max_bin),
        min_data_in_bin = as.integer(job$min_data_in_bin)
      ),
      free_raw_data = TRUE
    )

    params <- list(
      objective = job$objective,
      metric = "auc",
      boosting = job$boosting,
      num_threads = 1L,
      seed = as.integer(job$seed),
      verbosity = -1,
      learning_rate = job$learning_rate,
      num_leaves = as.integer(job$num_leaves),
      max_depth = as.integer(job$max_depth),
      min_data_in_leaf = as.integer(job$min_data_in_leaf),
      feature_fraction = job$feature_fraction,
      feature_fraction_bynode = job$feature_fraction_bynode,
      bagging_fraction = job$bagging_fraction,
      bagging_freq = as.integer(job$bagging_freq),
      lambda_l1 = job$lambda_l1,
      lambda_l2 = job$lambda_l2,
      min_gain_to_split = job$min_gain_to_split,
      min_sum_hessian_in_leaf = job$min_sum_hessian_in_leaf,
      scale_pos_weight = job$scale_pos_weight,
      is_unbalance = isTRUE(job$is_unbalance),
      boost_from_average = isTRUE(job$boost_from_average),
      extra_trees = isTRUE(job$extra_trees),
      path_smooth = job$path_smooth,
      first_metric_only = TRUE,
      feature_pre_filter = FALSE
    )
    if (isTRUE(job$force_col_wise)) {
      params$force_col_wise <- TRUE
    } else {
      params$force_row_wise <- TRUE
    }
    if (identical(job$boosting, "dart")) {
      params$drop_rate <- job$drop_rate
      params$skip_drop <- job$skip_drop
      params$max_drop <- as.integer(job$max_drop)
    }
    if (job$pos_bagging_fraction < 1 || job$neg_bagging_fraction < 1) {
      params$pos_bagging_fraction <- job$pos_bagging_fraction
      params$neg_bagging_fraction <- job$neg_bagging_fraction
      if (params$bagging_freq == 0L) params$bagging_freq <- 1L
    }

    nrounds <- as.integer(job$num_iterations)
    if (es > 0L && !is.null(dval)) {
      dvalid <- lightgbm::lgb.Dataset(
        data = data.matrix(dval[, campos, with = FALSE]),
        label = lab(dval$clase_ternaria),
        reference = ds
      )
      modelo <- lightgbm::lgb.train(
        params = params,
        data = ds,
        nrounds = nrounds,
        valids = list(valid = dvalid),
        early_stopping_rounds = es,
        verbose = -1
      )
    } else {
      modelo <- lightgbm::lgb.train(
        params = params,
        data = ds,
        nrounds = nrounds,
        verbose = -1
      )
    }

    prob <- predict(modelo, data.matrix(dtest[, campos, with = FALSE]))
    clase <- dtest$clase_ternaria
    gan <- sum(ifelse(
      prob > (1 / 40),
      ifelse(clase == "BAJA+2", 975000, -25000),
      0
    ))
    gan_norm <- gan / 0.30

    ord <- order(prob, decreasing = TRUE)
    cs <- cumsum(ifelse(clase[ord] == "BAJA+2", 975000, -25000))
    k_star <- as.integer(which.max(c(0, cs)) - 1L)
    k_full <- as.integer(round(k_star / 0.30))
    best_iter <- tryCatch(as.integer(modelo$best_iter), error = function(e) nrounds)
    if (length(best_iter) != 1L || is.na(best_iter)) best_iter <- nrounds

    data.table::data.table(
      trial_id = job$trial_id,
      familia = job$familia,
      param_optim = job$param_optim,
      semilla_split = job$semilla_split,
      semilla2 = job$semilla2,
      seed = job$seed,
      undersampling = job$undersampling,
      label_mode = job$label_mode,
      boosting = job$boosting,
      objective = job$objective,
      boost_from_average = job$boost_from_average,
      force_col_wise = job$force_col_wise,
      is_unbalance = job$is_unbalance,
      extra_trees = job$extra_trees,
      num_iterations = nrounds,
      best_iter = best_iter,
      learning_rate = job$learning_rate,
      feature_fraction = job$feature_fraction,
      feature_fraction_bynode = job$feature_fraction_bynode,
      min_data_in_leaf = job$min_data_in_leaf,
      num_leaves = job$num_leaves,
      max_depth = job$max_depth,
      lambda_l1 = job$lambda_l1,
      lambda_l2 = job$lambda_l2,
      min_gain_to_split = job$min_gain_to_split,
      bagging_fraction = job$bagging_fraction,
      bagging_freq = job$bagging_freq,
      pos_bagging_fraction = job$pos_bagging_fraction,
      neg_bagging_fraction = job$neg_bagging_fraction,
      min_sum_hessian_in_leaf = job$min_sum_hessian_in_leaf,
      scale_pos_weight = job$scale_pos_weight,
      drop_rate = job$drop_rate,
      skip_drop = job$skip_drop,
      max_drop = job$max_drop,
      max_bin = job$max_bin,
      min_data_in_bin = job$min_data_in_bin,
      path_smooth = job$path_smooth,
      early_stopping_rounds = es,
      ganancia = gan_norm,
      k_star = k_star,
      mejores_envios = k_full,
      tiempo_seg = round(as.numeric(difftime(Sys.time(), t0, units = "secs")), 1),
      error = ""
    )
  }, error = function(e) {
    data.table::data.table(
      trial_id = job$trial_id,
      familia = job$familia,
      param_optim = job$param_optim,
      ganancia = NA_real_,
      error = conditionMessage(e)
    )
  })
}


## 4. Los experimentos

Centro nuevo: hojas 8, `lr=0.027`, 1000 rondas, `min_data=76`, `max_bin=127`, undersampling 1. Cada config x 3 semillas fijas. El rankeo (seccion 6) usa la media.


In [ ]:
# +++ centro del diseno. Cada familia pisa UN eje (o un par acoplado) sobre este base.
# +++ No es el grid de z494 (AUC, 3 params). La metrica la calcula eval_job.
base <- list(
  boosting = "gbdt",
  objective = "binary",
  boost_from_average = TRUE,
  force_col_wise = FALSE,
  is_unbalance = FALSE,
  extra_trees = FALSE,
  num_iterations = 1000L,
  learning_rate = 0.027,
  feature_fraction = 0.8,
  feature_fraction_bynode = 1.0,
  min_data_in_leaf = 76L,
  num_leaves = 8L,
  max_depth = -1L,
  lambda_l1 = 0,
  lambda_l2 = 0,
  min_gain_to_split = 0,
  bagging_fraction = 1.0,
  bagging_freq = 0L,
  pos_bagging_fraction = 1.0,
  neg_bagging_fraction = 1.0,
  min_sum_hessian_in_leaf = 0.001,
  scale_pos_weight = 1,
  drop_rate = 0.1,
  skip_drop = 0.5,
  max_drop = 50L,
  max_bin = 127L,
  min_data_in_bin = 3L,
  path_smooth = 0,
  early_stopping_rounds = 0L,
  undersampling = 1.0,
  label_mode = "baja2",
  semilla_split = PARAM$semilla_primigenia,
  semilla2 = PARAM$semilla2,
  seed = PARAM$semilla_primigenia
)

fix_job <- function(job) {
  if (isTRUE(job$bagging_fraction < 1) && job$bagging_freq == 0L) job$bagging_freq <- 1L
  if (isTRUE(job$pos_bagging_fraction < 1 || job$neg_bagging_fraction < 1) && job$bagging_freq == 0L) {
    job$bagging_freq <- 1L
  }
  if (isTRUE(job$is_unbalance)) job$scale_pos_weight <- 1
  if (job$max_depth > 0 && job$num_leaves > (2^job$max_depth - 1)) {
    job$num_leaves <- as.integer(2^job$max_depth - 1)
  }
  if (identical(job$boosting, "dart") && job$num_iterations > 200L) {
    job$num_iterations <- 200L  # +++ dart es mucho mas lento; tope para que el barrido termine
  }
  job
}

one <- function(familia, param_optim, overrides) {
  job <- modifyList(base, overrides)
  job$familia <- familia
  job$param_optim <- param_optim
  fix_job(job)
}

expand_jobs <- function(familia, param_optim, grid) {
  tb <- do.call(CJ, grid)
  lapply(seq_len(nrow(tb)), function(i) {
    ov <- as.list(tb[i])
    one(familia, param_optim, ov)
  })
}

diagonal <- function(familia, param_optim, cols) {
  n <- length(cols[[1]])
  lapply(seq_len(n), function(i) {
    ov <- lapply(cols, function(v) v[[i]])
    one(familia, param_optim, ov)
  })
}

jobs <- list()
push <- function(xs) jobs <<- c(jobs, xs)

# --- referencias (puntos nombrados, no un barrido) ---
push(list(one(
  "00_baseline", "baseline del diseno",
  list()
)))
push(list(one(
  "00_ref_z102", "punto z102 (curso)",
  list(learning_rate = 0.05, num_iterations = 100L, num_leaves = 31L,
       min_data_in_leaf = 100L, feature_fraction = 0.5, max_bin = 31L, undersampling = 1)
)))
push(list(one(
  "00_ref_denicolay", "punto Denicolay planilla",
  list(num_iterations = 1000L, learning_rate = 0.027, feature_fraction = 0.8,
       min_data_in_leaf = 76L, num_leaves = 8L, max_depth = -1L, undersampling = 1)
)))

# --- centro = zona que en z495 no era ruido: pocas hojas, lr chico, datos completos ---
# +++ el 618M de "semilla" era el mismo modelo con otro split. Aca cada config se corre
# +++ en las MISMAS 3 semillas y se rankea por el promedio.

push(list(one("00_centro", "centro Denicolay empujado (us=1, leaves=8, lr=0.027, bin=127)", list())))
push(list(one(
  "00_ref_kaggle5940_style", "referencia: hojas medias, us=1, bin=31 (estilo del 375.9)",
  list(num_leaves = 31L, learning_rate = 0.05, num_iterations = 500L,
       min_data_in_leaf = 100L, feature_fraction = 0.8, max_bin = 31L,
       undersampling = 1, max_depth = -1L)
)))

# extremos de complejidad, alrededor de hojas chicas
push(expand_jobs("num_leaves", "num_leaves", list(
  num_leaves = c(2L, 3L, 4L, 6L, 8L, 12L, 16L, 24L, 32L, 48L, 64L)
)))
push(expand_jobs("min_data_in_leaf", "min_data_in_leaf", list(
  min_data_in_leaf = c(5L, 15L, 30L, 50L, 60L, 76L, 90L, 120L, 180L, 300L, 600L, 1200L)
)))
push(expand_jobs("max_depth", "max_depth", list(
  max_depth = c(-1L, 2L, 3L, 4L, 5L, 6L, 8L, 12L)
)))
push(expand_jobs("min_gain_to_split", "min_gain_to_split", list(
  min_gain_to_split = c(0, 0.01, 0.1, 1, 5, 20, 50)
)))
push(expand_jobs("min_sum_hessian_in_leaf", "min_sum_hessian_in_leaf", list(
  min_sum_hessian_in_leaf = c(1e-3, 0.1, 1, 10, 50, 200)
)))

# lr mas chico y muchas mas rondas (el extremo que Denicolay ya insinuaba)
push(diagonal(
  "lr_x_nrounds",
  "learning_rate + num_iterations extremos",
  list(
    learning_rate = c(0.003, 0.005, 0.01, 0.015, 0.02, 0.027, 0.04, 0.06, 0.1),
    num_iterations = c(5000L, 3000L, 2000L, 1500L, 1200L, 1000L, 600L, 350L, 200L)
  )
))
push(expand_jobs("num_iterations", "num_iterations (lr fijo 0.02)", list(
  learning_rate = c(0.02),
  num_iterations = c(400L, 800L, 1500L, 2500L, 4000L)
)))

# feature_fraction: el joint bueno uso 0.3 y 0.9; empujar ambos lados
push(expand_jobs("feature_fraction", "feature_fraction", list(
  feature_fraction = c(0.08, 0.15, 0.25, 0.4, 0.6, 0.8, 0.9, 0.97, 1.0)
)))
push(expand_jobs("feature_fraction_bynode", "feature_fraction_bynode", list(
  feature_fraction_bynode = c(0.2, 0.4, 0.6, 0.8, 1.0)
)))

# regularizacion mas alta
push(expand_jobs("lambda_l1", "lambda_l1", list(
  lambda_l1 = c(0, 0.1, 1, 3, 10, 30, 100)
)))
push(expand_jobs("lambda_l2", "lambda_l2", list(
  lambda_l2 = c(0, 0.1, 1, 10, 50, 200)
)))

# bagging fue la familia ESTABLE en z495 (media alta, rango chico). Empujar fraction chica.
push(expand_jobs("bagging", "bagging_fraction + bagging_freq", list(
  bagging_fraction = c(0.1, 0.2, 0.3, 0.45, 0.6, 0.8),
  bagging_freq = c(1L, 5L)
)))
push(expand_jobs("pos_neg_bagging", "pos/neg bagging", list(
  pos_bagging_fraction = c(1.0, 0.6),
  neg_bagging_fraction = c(0.15, 0.3, 0.5),
  bagging_freq = c(1L)
)))

# desbalance: is_unbalance en z495 destruyo (122M). No se repite.
# undersampling: los trials buenos tenian 1.0, no 0.02
push(expand_jobs("undersampling", "undersampling (lado alto)", list(
  undersampling = c(0.4, 0.6, 0.8, 0.9, 1.0)
)))
push(expand_jobs("scale_pos_weight", "scale_pos_weight (rango corto)", list(
  scale_pos_weight = c(1, 2, 4, 8)
)))

# max_bin 255 aparecio varias veces en el top. Subir hasta 1023.
push(expand_jobs("max_bin", "max_bin", list(
  max_bin = c(31L, 63L, 127L, 255L, 511L, 1023L)
)))
push(expand_jobs("min_data_in_bin", "min_data_in_bin", list(
  min_data_in_bin = c(1L, 3L, 10L, 30L, 80L)
)))

push(expand_jobs("extra_trees", "extra_trees", list(extra_trees = c(TRUE))))
push(expand_jobs("max_depth_con_hojas_chicas", "max_depth con num_leaves=8", list(
  max_depth = c(3L, 4L, 6L, 10L, -1L),
  num_leaves = c(8L)
)))

# interacciones en la zona que pago
push(expand_jobs("ix_leaves_x_mindata", "num_leaves x min_data", list(
  num_leaves = c(4L, 8L, 16L, 32L),
  min_data_in_leaf = c(30L, 76L, 150L, 400L)
)))
push(expand_jobs("ix_leaves_x_bin", "num_leaves x max_bin", list(
  num_leaves = c(4L, 8L, 16L),
  max_bin = c(63L, 255L, 511L)
)))
push(expand_jobs("ix_ff_x_leaves", "feature_fraction x num_leaves", list(
  feature_fraction = c(0.2, 0.4, 0.8, 1.0),
  num_leaves = c(4L, 8L, 16L, 32L)
)))
push(expand_jobs("ix_bag_x_leaves", "bagging x num_leaves", list(
  bagging_fraction = c(0.2, 0.5, 0.8),
  bagging_freq = c(1L),
  num_leaves = c(8L, 16L, 32L)
)))
push(expand_jobs("ix_l1_x_leaves", "lambda_l1 x num_leaves", list(
  lambda_l1 = c(0, 1, 10),
  num_leaves = c(4L, 8L, 16L)
)))
push(diagonal(
  "ix_lr_largo",
  "lr muy chico x muchas rondas x hojas chicas",
  list(
    learning_rate = c(0.005, 0.01, 0.015, 0.02),
    num_iterations = c(4000L, 2500L, 2000L, 1500L),
    num_leaves = c(4L, 8L, 8L, 12L),
    max_bin = c(255L, 255L, 127L, 255L)
  )
))

# conjunta SOLO en la caja que no exploto en z495
sample_joint <- function(n, familia, param_optim, seed) {
  set.seed(seed)
  out <- vector("list", n)
  for (i in seq_len(n)) {
    md <- sample(c(-1L, 3L, 4L, 6L, 8L), 1)
    nl <- sample(c(3L, 4L, 6L, 8L, 12L, 16L, 24L), 1)
    if (md > 0) nl <- min(nl, as.integer(2^md - 1))
    bf <- sample(c(1, 0.8, 0.5, 0.3, 0.15), 1)
    out[[i]] <- one(familia, param_optim, list(
      learning_rate = round(10^runif(1, log10(0.005), log10(0.05)), 4),
      num_iterations = sample(c(800L, 1200L, 1800L, 2500L, 4000L), 1),
      num_leaves = nl,
      max_depth = md,
      min_data_in_leaf = sample(c(20L, 40L, 60L, 76L, 100L, 150L, 250L), 1),
      feature_fraction = sample(c(0.15, 0.3, 0.5, 0.7, 0.85, 1), 1),
      lambda_l1 = sample(c(0, 0, 0.1, 1, 5), 1),
      lambda_l2 = sample(c(0, 0, 1, 5), 1),
      bagging_fraction = bf,
      bagging_freq = if (bf < 1) 1L else 0L,
      undersampling = sample(c(0.8, 1, 1, 1), 1),
      max_bin = sample(c(63L, 127L, 255L, 511L), 1),
      scale_pos_weight = 1
    ))
  }
  out
}
push(sample_joint(100L, "joint_extremo", "joint en la caja ganadora, empujada", PARAM$semilla_primigenia + 3L))

# +++ duplicados, despues x3 semillas FIJAS (mismo trio para todas las configs)
sig_of <- function(job) {
  nms <- sort(setdiff(names(job), c("trial_id", "familia", "param_optim", "semilla_split", "semilla2", "seed")))
  paste(vapply(nms, function(nm) paste0(nm, "=", paste(job[[nm]], collapse = ",")), character(1)), collapse = "|")
}
seen <- character()
jobs_u <- list()
for (job in jobs) {
  s <- sig_of(job)
  if (s %in% seen) next
  seen <- c(seen, s)
  jobs_u[[length(jobs_u) + 1L]] <- job
}
SEMILLAS <- c(271211L, 200177L, 410551L)
jobs <- list()
for (job in jobs_u) {
  for (sv in SEMILLAS) {
    j <- job
    j$semilla_split <- sv
    j$seed <- sv
    j$semilla2 <- sv + 17L
    jobs[[length(jobs) + 1L]] <- j
  }
}
for (i in seq_along(jobs)) jobs[[i]]$trial_id <- i

cat("configs:", length(jobs_u), "  jobs (x3 semillas):", length(jobs), "\n")
print(rbindlist(lapply(jobs_u, function(j) data.table(familia = j$familia)))[, .N, by = familia][order(-N)])


## 5. Corrida

PSOCK + 1 thread por worker. Progreso por tanda. Si un modelo se cae (parametro no soportado por esa version de lightgbm) queda en `error` y el resto sigue.


In [ ]:
archivo_trials <- "trials_th04.tsv"
tb_trials <- data.table()
done_ids <- integer()
if (file.exists(archivo_trials)) {
  tb_trials <- fread(archivo_trials)
  if ("ganancia" %in% names(tb_trials)) {
    done_ids <- tb_trials[is.finite(ganancia), unique(trial_id)]
  }
}
pending <- Filter(function(j) !(j$trial_id %in% done_ids), jobs)
cat("pendientes:", length(pending), " de ", length(jobs), "\n", sep = "")

if (length(pending) > 0L) {
  cl <- makeCluster(PARAM$mc_cores, type = "PSOCK")
  ok_cluster <- TRUE
  clusterEvalQ(cl, {
    Sys.setenv(OMP_NUM_THREADS = "1", MKL_NUM_THREADS = "1")
    suppressPackageStartupMessages({
      library(data.table)
      library(lightgbm)
    })
    data.table::setDTthreads(1)
    NULL
  })
  assign("eval_job", eval_job, envir = .GlobalEnv)
  assign("dataset_mes", dataset_mes, envir = .GlobalEnv)
  clusterExport(cl, c("eval_job", "dataset_mes"), envir = .GlobalEnv)

  bs <- PARAM$mc_cores
  n_b <- ceiling(length(pending) / bs)
  t0 <- Sys.time()
  for (b in seq_len(n_b)) {
    idx <- ((b - 1L) * bs + 1L):min(b * bs, length(pending))
    res <- parLapply(cl, pending[idx], eval_job)
    tb_new <- rbindlist(res, fill = TRUE)
    tb_trials <- rbindlist(list(tb_trials, tb_new), fill = TRUE)
    fwrite(tb_trials, archivo_trials, sep = "\t")

    n_ok <- tb_new[is.finite(ganancia), .N]
    n_bad <- nrow(tb_new) - n_ok
    mins <- as.numeric(difftime(Sys.time(), t0, units = "mins"))
    cat(sprintf(
      "tanda %d/%d | ok %d | fallos %d | mejor tanda %s | %.1f min | ETA %.1f min\n",
      b, n_b, n_ok, n_bad,
      if (n_ok) format(max(tb_new$ganancia, na.rm = TRUE), big.mark = ",", scientific = FALSE) else "NA",
      mins, if (b < n_b) mins / b * (n_b - b) else 0
    ))
    malos <- tb_new[!is.finite(ganancia) | nzchar(error)]
    if (nrow(malos)) print(malos[, .(trial_id, familia, error)])
    flush.console()
  }
  stopCluster(cl)
  ok_cluster <- FALSE
}

cat("trials con ganancia:", tb_trials[is.finite(ganancia), .N], "\n")


## 6. Que parametro movio la ganancia, y la fila de la planilla

`delta_vs_baseline` > 0: esa familia encontro algo mejor que el centro. Si el rango de una familia es chico, ese parametro no vale la pena tunearlo en la proxima pasada.

`planilla_TH04.tsv` tiene una fila por familia, con el mejor punto de esa familia, en el orden de la hoja.


In [ ]:
# +++ rankeo por PROMEDIO de las 3 semillas, no por el maximo de una.
# +++ en z495 el "ganador" (618M) era la config del centro con un split suertudo;
# +++ la misma config en Kaggle perdio contra KA5940 (375.9 en el cupo 10500).
tb <- tb_trials[is.finite(ganancia)]
ignorar <- c(
  "trial_id", "semilla_split", "semilla2", "seed", "ganancia", "k_star",
  "mejores_envios", "tiempo_seg", "error", "best_iter"
)
cols <- setdiff(names(tb), ignorar)
tb[, cfg := apply(.SD, 1, paste, collapse = "|"), .SDcols = cols]

agg <- tb[, .(
  ganancia = mean(ganancia),
  ganancia_sd = sd(ganancia),
  ganancia_max = max(ganancia),
  n_seeds = .N,
  mejores_envios = as.integer(round(mean(mejores_envios)))
), by = c("cfg", "familia", "param_optim")]

centro <- agg[familia == "00_baseline", ganancia][1]
agg[, delta_vs_centro := ganancia - centro]
setorder(agg, -ganancia)
fwrite(agg, "configs_th04.tsv", sep = "\t")

cat("centro (media 3 semillas):", format(centro, big.mark = ",", scientific = FALSE), "\n\n")
cat("=== top 20 CONFIGS por media (no por una semilla) ===\n")
print(agg[1:min(20, .N), .(familia, ganancia, ganancia_sd, n_seeds, mejores_envios, delta_vs_centro)])

por_familia <- agg[, .(
  n = .N,
  ganancia_max = max(ganancia),
  ganancia_mean = mean(ganancia)
), by = .(familia, param_optim)]
por_familia[, delta_vs_centro := ganancia_max - centro]
setorder(por_familia, -ganancia_max)
fwrite(por_familia, "sensibilidad_th04.tsv", sep = "\t")
cat("\n=== mejor config de cada familia ===\n")
print(por_familia)

# +++ fila de planilla = mejor config de la familia (media), con sus hiperparametros
best_cfg <- agg[, .SD[which.max(ganancia)], by = familia]
rep_row <- tb[best_cfg, on = "cfg", mult = "first"]
rep_row[, ganancia := i.ganancia]
rep_row[, mejores_envios := i.mejores_envios]
n_map <- por_familia[, .(familia, n_fam = n, ganancia_mean_fam = ganancia_mean)]
rep_row <- n_map[rep_row, on = "familia"]

planilla <- rep_row[, .(
  semilla_primigenia = PARAM$semilla_primigenia,
  `semilla2 (undersampling)` = semilla2,
  `Porción Undesampling` = undersampling,
  `Param a optimizar` = param_optim,
  `# semillas en BO` = n_seeds,
  `BO iterations` = n_fam,
  num_iterations = num_iterations,
  learning_rate = learning_rate,
  feature_fraction = feature_fraction,
  min_data_in_leaf = min_data_in_leaf,
  num_leaves = num_leaves,
  max_depth = max_depth,
  lambda_l1 = lambda_l1,
  lambda_l2 = lambda_l2,
  min_gain_to_split = min_gain_to_split,
  bagging_fraction = bagging_fraction,
  min_sum_hessian_in_leaf = min_sum_hessian_in_leaf,
  bagging_freq = bagging_freq,
  scale_pos_weight = scale_pos_weight,
  boosting = boosting,
  boost_from_average = boost_from_average,
  objective = objective,
  first_metric_only = TRUE,
  drop_rate = drop_rate,
  skip_drop = skip_drop,
  force_col_wise = force_col_wise,
  is_unbalance = is_unbalance,
  max_drop = max_drop,
  max_bin = max_bin,
  n_estimators = num_iterations,
  early_stopping_rounds = early_stopping_rounds,
  min_child_weight = min_sum_hessian_in_leaf,
  feature_fraction_bynode = feature_fraction_bynode,
  bagging_freq_2 = bagging_freq,
  Prueba = ganancia,
  `mejores envios` = mejores_envios,
  `Public Leaderboard` = NA_real_,
  `Promedio de las pruebas` = ganancia_mean_fam,
  Estudiante = PARAM$estudiante
)]
setorder(planilla, -Prueba)
fwrite(planilla, "planilla_TH04.tsv", sep = "\t")
cat("\nplanilla:", nrow(planilla), "filas\n")
print(planilla[1:min(8, .N), .(`Param a optimizar`, Prueba, `mejores envios`, num_leaves, learning_rate, num_iterations, feature_fraction, max_bin, undersampling)])

# +++ el objeto tb que usa el submit es la mejor CONFIG (media), no un split suelto
tb <- rep_row[which.max(ganancia)]
cat("\nA submitear (media 3 semillas):", tb$familia, format(tb$ganancia, big.mark = ",", scientific = FALSE),
    " cupo~", tb$mejores_envios, "\n")


## 7. Submit localizado del ganador (opcional)

Entrena sobre **todo** 202107, sin undersampling. `min_data_in_leaf` se escala por `1/undersampling`, como hace `z494`, porque el hiperparametro se eligio sobre una muestra mas chica.

Genera los CSV de varios cupos. Submitea solo 3 para no quemar el limite diario. Despues copia el `publicScore` a la columna Public Leaderboard de esa fila.


In [ ]:
CORRER_KAGGLE <- FALSE  # +++ pasar a TRUE cuando ya viste el ganador local
PARAM$submit_cortes <- c(10500L, 11000L, 11500L)

if (CORRER_KAGGLE) {
  mejor <- tb[which.max(ganancia)]
  print(mejor[, .(familia, ganancia, num_leaves, learning_rate, num_iterations, min_data_in_leaf, feature_fraction, undersampling)])

  dfull <- dataset[foto_mes == 202107]
  dfuture <- dataset[foto_mes == 202109]
  y <- as.integer(dfull$clase_ternaria == "BAJA+2")
  drop_cols <- c("clase_ternaria", "numero_de_cliente", "foto_mes", "clase01", "azar", "training")
  campos <- setdiff(colnames(dfull), drop_cols)
  campos <- campos[vapply(dfull[, campos, with = FALSE], is.numeric, logical(1))]

  # +++ z494: el min_data hallado sobre la muestra undersampleada se reescala al mes completo
  min_data_final <- as.integer(round(mejor$min_data_in_leaf / mejor$undersampling))
  cat("min_data_in_leaf final:", min_data_final, "\n")

  ds <- lgb.Dataset(
    data = data.matrix(dfull[, campos, with = FALSE]),
    label = y,
    params = list(max_bin = as.integer(mejor$max_bin)),
    free_raw_data = TRUE
  )
  params <- list(
    objective = "binary", metric = "auc", boosting = mejor$boosting,
    num_threads = PARAM$mc_cores, seed = PARAM$semilla_primigenia, verbosity = -1,
    learning_rate = mejor$learning_rate,
    num_leaves = as.integer(mejor$num_leaves),
    max_depth = as.integer(mejor$max_depth),
    min_data_in_leaf = min_data_final,
    feature_fraction = mejor$feature_fraction,
    feature_fraction_bynode = mejor$feature_fraction_bynode,
    lambda_l1 = mejor$lambda_l1, lambda_l2 = mejor$lambda_l2,
    min_gain_to_split = mejor$min_gain_to_split,
    min_sum_hessian_in_leaf = mejor$min_sum_hessian_in_leaf,
    bagging_fraction = mejor$bagging_fraction,
    bagging_freq = as.integer(mejor$bagging_freq),
    scale_pos_weight = mejor$scale_pos_weight,
    is_unbalance = isTRUE(mejor$is_unbalance),
    boost_from_average = isTRUE(mejor$boost_from_average),
    extra_trees = isTRUE(mejor$extra_trees),
    path_smooth = mejor$path_smooth,
    feature_pre_filter = FALSE,
    force_row_wise = TRUE
  )
  modelo <- lgb.train(
    params = params, data = ds,
    nrounds = as.integer(if (mejor$best_iter > 0) mejor$best_iter else mejor$num_iterations),
    verbose = -1
  )
  prob <- predict(modelo, data.matrix(dfuture[, campos, with = FALSE]))
  ord <- order(prob, decreasing = TRUE)

  for (envios in seq(10000L, 12000L, by = 500L)) {
    pred <- integer(length(prob))
    pred[ord[seq_len(envios)]] <- 1L
    archivo <- sprintf("KA496_%05d.csv", envios)
    fwrite(data.table(numero_de_cliente = dfuture$numero_de_cliente, Predicted = pred), archivo)
    cat("csv", archivo, "\n")
    if (envios %in% PARAM$submit_cortes) {
      linea <- sprintf(
        "kaggle competitions submit -c labo-1-ba-inicial -f %s -m 'TH04 %s envios=%d leaves=%s lr=%s ff=%s'",
        archivo, mejor$familia, envios, mejor$num_leaves, mejor$learning_rate, mejor$feature_fraction
      )
      salida <- system(linea, intern = TRUE)
      cat("  SUBMIT:", salida, "\n")
    }
  }
  flush.console()
}


In [ ]:
# +++ scores de Kaggle, para copiar a Public Leaderboard
if (CORRER_KAGGLE) {
  cat(system("kaggle competitions submissions -c labo-1-ba-inicial", intern = TRUE), sep = "\n")
}
